# Step 4: Data Cleaning and Wrangling

## Purpose
This notebook makes the Telco churn dataset usable and consistent for later modeling and interpretation.

## Step 4 questions
- How do we fix missing or blank values?
- Which data types are incorrect?
- Are there duplicate customers?
- Are there inconsistent labels?
- Do numeric features contain suspicious outliers?

## Expected outputs
- A cleaned dataset
- Reproducible cleaning logic
- A clear record of cleaning choices and trade-offs

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

In [12]:
PROJECT_DIR = Path.cwd()
DATA_FILE = PROJECT_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
CLEANED_FILE = PROJECT_DIR / "WA_Fn-UseC_-Telco-Customer-Churn-cleaned.csv"

df_raw = pd.read_csv(DATA_FILE)
df_raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Load the raw data and inspect its basic structure

We begin with the raw CSV exactly as provided so the cleaning process remains transparent.

df_raw.shape, df_raw.columns.tolist()

In [13]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 2. Diagnose raw-data quality issues

Before cleaning, we check for missing values, blank strings, duplicate customer records, and label consistency.

In [14]:
missing_values = df_raw.isnull().sum().sort_values(ascending=False)
blank_strings = (
    df_raw.select_dtypes(include='object')
    .apply(lambda column: column.astype(str).str.strip().eq('').sum())
    .sort_values(ascending=False)
)
duplicate_customer_ids = df_raw.duplicated(subset='customerID').sum()

print('Missing values by column:')
print(missing_values)
print()
print('Blank strings in object columns:')
print(blank_strings)
print()
print(f'Duplicate customerID rows: {duplicate_customer_ids}')
print()
print("Unique values in 'Churn':")
print(df_raw['Churn'].value_counts(dropna=False))

Missing values by column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Blank strings in object columns:
TotalCharges        11
customerID           0
gender               0
Partner              0
PhoneService         0
Dependents           0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
MultipleLines        0
DeviceProtection     0
TechSupport          0
StreamingMovies      0
StreamingTV          0
Contract             0
PaperlessBilling     0
PaymentMethod        0
Churn                0
dtype: int64

Duplicate customerID rows: 0

Unique 

## 3. Clean the dataset reproducibly

Our cleaning plan for this dataset is:

- remove leading/trailing spaces in text columns
- convert `TotalCharges` to numeric
- remove duplicate customer records if any exist
- inspect rows where `TotalCharges` becomes missing after conversion
- drop rows with unresolved missing `TotalCharges`
- keep the cleaning logic explicit and reproducible

In [15]:
def clean_telco_data(df: pd.DataFrame) -> pd.DataFrame:
    cleaned_df = df.copy()

    object_columns = cleaned_df.select_dtypes(include='object').columns
    for column in object_columns:
        cleaned_df[column] = cleaned_df[column].astype(str).str.strip()

    cleaned_df['TotalCharges'] = pd.to_numeric(cleaned_df['TotalCharges'], errors='coerce')
    cleaned_df = cleaned_df.drop_duplicates(subset='customerID').copy()
    cleaned_df = cleaned_df.dropna(subset=['TotalCharges']).copy()

    return cleaned_df


df_cleaned = clean_telco_data(df_raw)

print('Raw shape:', df_raw.shape)
print('Cleaned shape:', df_cleaned.shape)
print('Rows removed:', len(df_raw) - len(df_cleaned))

Raw shape: (7043, 21)
Cleaned shape: (7032, 21)
Rows removed: 11


In [16]:
print('Cleaned dataset info:')
df_cleaned.info()
print()
print('Missing values after cleaning:')
print(df_cleaned.isnull().sum())
print()
print('Duplicate customerID rows after cleaning:', df_cleaned.duplicated(subset='customerID').sum())
print()
print("Target distribution after cleaning:")
print(df_cleaned['Churn'].value_counts(dropna=False))

Cleaned dataset info:
<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 no

## 4. Review numeric columns after cleaning

At this stage, we inspect the cleaned numeric columns. We are not aggressively removing outliers yet because unusual customer values may still carry real churn signal.

In [17]:
numeric_columns = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
df_cleaned[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7032.0,0.162400,0.368844,0.00,0.0000,0.000,0.0000,1.00
tenure,7032.0,32.421786,24.545260,1.00,9.0000,29.000,55.0000,72.00
MonthlyCharges,7032.0,64.798208,30.085974,18.25,35.5875,70.350,89.8625,118.75
TotalCharges,7032.0,2283.300441,2266.771362,18.80,401.4500,1397.475,3794.7375,8684.80


In [18]:
df_cleaned.to_csv(CLEANED_FILE, index=False)
print(f'Cleaned dataset saved to: {CLEANED_FILE}')

Cleaned dataset saved to: c:\Users\HP\Documents\Google drive documents\Beginnings\Data Science ML Model buidling\Projects\Telco-Customer-Churn\WA_Fn-UseC_-Telco-Customer-Churn-cleaned.csv


In [19]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 
